In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

df1 = pd.read_csv('/content/drive/MyDrive/UAIN1008R_Proyecto/datos/raw/Red vial nacional del Sistema Nacional de Carreteras 2022 a 2024 - Ministerio de Transportes y Comunicaciones – MTC.csv', encoding = 'latin1', sep=';')
df2 = pd.read_csv('/content/drive/MyDrive/UAIN1008R_Proyecto/datos/raw/Dataset GEI CSV_0.csv', sep=';')
df3 = pd.read_csv('/content/drive/MyDrive/UAIN1008R_Proyecto/datos/raw/Información Trafico - Carga y Operaciones - LAP Ene 2019 - Dic 2025 - Organismo Supervisor de la Inversión en Infraestructura de Transporte de Uso Público – OSITRAN.csv')
df4 = pd.read_csv('/content/drive/MyDrive/UAIN1008R_Proyecto/datos/raw/Transporte Terrestre Carga Internacional_2023-2024.csv')

Mounted at /content/drive


# Extracción de Dataset 1

In [2]:
#Tratamiento de Dataset 1
print("===== Tratamiento de Datasets ====")
print("\nPerfilado del Dataset 1 antes:")

# Métricas antes de la limpieza:
print(f"Nulos: {df1.isnull().sum().sum()}")
print(f"Duplicados: {df1.duplicated().sum()}")

# Identificación de outliders por IQR
numericos1 = df1.select_dtypes(include='number')

q1_1 = numericos1.quantile(0.25)
q3_1 = numericos1.quantile(0.75)
iqr_1 = q3_1 - q1_1
outliers_1 = ((numericos1 < (q1_1 - 1.5 * iqr_1)) | (numericos1 > (q3_1 + 1.5 * iqr_1))).sum()

print("\nNúmero de outliers por columna:")
outliers_1

===== Tratamiento de Datasets ====

Perfilado del Dataset 1 antes:
Nulos: 0
Duplicados: 0

Número de outliers por columna:


,0
ID_RVN,0
ID_DEPARTAMENTO,0
INICIO,3940
FINAL,3932
LONGITUD,2576
NRO_CARRILES,5223
SUPERFICIE,0
ESTADO,185
CODIGO_CONCESION,4916
FECHA_CORTE,0


# Tratamiento de Dataset 1

In [3]:
df1_limpio = df1.copy()
print("\n1. Normalización de formato: ")

# Reajuste Del formato y tipo de la columna FECHA_CORTE
print(f"\n1.1. FECHA_CORTE (Antes del cambio): {df1_limpio['FECHA_CORTE'].dtype}")
print(f"Formato de FECHA_CORTE (Antes del cambio):\n{df1_limpio['FECHA_CORTE']}")

# Asignación temporal del tipo str para facilitar conversión
df1_limpio['FECHA_CORTE'] = pd.to_datetime(df1_limpio['FECHA_CORTE'].astype(str), format='%Y%m%d', errors='coerce')
print(f"\n1.2. FECHA_CORTE (Después del cambio): {df1_limpio["FECHA_CORTE"].dtype}")
print(f"Formato de FECHA_CORTE (Después del cambio):\n{df1_limpio["FECHA_CORTE"]}\n")


1. Normalización de formato: 

1.1. FECHA_CORTE (Antes del cambio): int64
Formato de FECHA_CORTE (Antes del cambio):
0        20221231
1        20221231
2        20221231
3        20221231
4        20221231
           ...   
27285    20241231
27286    20241231
27287    20241231
27288    20241231
27289    20241231
Name: FECHA_CORTE, Length: 27290, dtype: int64

1.2. FECHA_CORTE (Después del cambio): datetime64[ns]
Formato de FECHA_CORTE (Después del cambio):
0       2022-12-31
1       2022-12-31
2       2022-12-31
3       2022-12-31
4       2022-12-31
           ...    
27285   2024-12-31
27286   2024-12-31
27287   2024-12-31
27288   2024-12-31
27289   2024-12-31
Name: FECHA_CORTE, Length: 27290, dtype: datetime64[ns]



In [4]:
#Correción de Títulos
print("\n2. Correción de Títulos: ")
print(f"\nDataFrame original:")
print(df1_limpio[['CODIGO_CONCESION', "CODIGO_LOGISTICO"]])
print(f"\nDataFrame modificado:")
df1_limpio.rename(columns={'CODIGO_CONCESION': 'CODIGO_LOGISTICO', 'CODIGO_LOGISTICO': 'CODIGO_CONCESION'}, inplace=True)
print(df1_limpio[['CODIGO_CONCESION', "CODIGO_LOGISTICO"]])


2. Correción de Títulos: 

DataFrame original:
       CODIGO_CONCESION CODIGO_LOGISTICO
0                     2              C02
1                     2              C02
2                     2              C02
3                     7            ALC02
4                     7            ALC02
...                 ...              ...
27285               999               NC
27286               999               NC
27287               999               NC
27288               999               NC
27289               999               NC

[27290 rows x 2 columns]

DataFrame modificado:
      CODIGO_CONCESION  CODIGO_LOGISTICO
0                  C02                 2
1                  C02                 2
2                  C02                 2
3                ALC02                 7
4                ALC02                 7
...                ...               ...
27285               NC               999
27286               NC               999
27287               NC               999
2

In [5]:
# Unificar Superficie
print(f"\n3. SUPERFICIE_L antes del reemplazo:\n{df1_limpio['SUPERFICIE_L'].value_counts()}\n")
df1_limpio['SUPERFICIE_L'] = df1_limpio['SUPERFICIE_L'].replace('Proyectada', 'Proyectado')
df1_limpio['SUPERFICIE_L'] = df1_limpio['SUPERFICIE_L'].replace('Pavimentada', 'Pavimentado')
df1_limpio['SUPERFICIE_L'] = df1_limpio['SUPERFICIE_L'].replace('Afirmada', 'Afirmado')
print(f"\n4. SUPERFICIE_L luego del reemplazo:\n{df1_limpio['SUPERFICIE_L'].value_counts()}\n")


3. SUPERFICIE_L antes del reemplazo:
SUPERFICIE_L
Pavimentado            11594
Asfaltado económico     8100
Pavimentada             4042
Afirmado                1772
Trocha                   682
Afirmada                 550
Sin afirmar              365
Proyectado               136
Proyectada                49
Name: count, dtype: int64


4. SUPERFICIE_L luego del reemplazo:
SUPERFICIE_L
Pavimentado            15636
Asfaltado económico     8100
Afirmado                2322
Trocha                   682
Sin afirmar              365
Proyectado               185
Name: count, dtype: int64



In [6]:
print("\n5. Tratamiento de outliders: ")

# Identificación de valores con registro -1
carriles_neg = (df1_limpio['NRO_CARRILES'] == -1).sum()
print(f"\nRegistros con '-1' (Antes del tratamiento):")
print(f"- En NRO_CARRILES: {carriles_neg}")

# Tratamiento de valores -1 (Eliminación)
df1_limpio = df1_limpio[df1_limpio['NRO_CARRILES'] != -1]

print("\nConteo de valores -1 (Después del tratamiento):")
print(f"- En NRO_CARRILES: {(df1_limpio['NRO_CARRILES'] == -1).sum()}")


5. Tratamiento de outliders: 

Registros con '-1' (Antes del tratamiento):
- En NRO_CARRILES: 4408

Conteo de valores -1 (Después del tratamiento):
- En NRO_CARRILES: 0


In [7]:
# Estadísticas descriptivas para columnas numéricas
df1_limpio.describe()

,ID_RVN,ID_DEPARTAMENTO,INICIO,FINAL,LONGITUD,NRO_CARRILES,SUPERFICIE,ESTADO,CODIGO_LOGISTICO,FECHA_CORTE,ES_CONCES
count,22882.000000,22882.000000,22882.000000,22882.000000,22882.000000,22882.000000,22882.000000,22882.000000,22882.000000,22882,22882.000000
mean,13571.739402,11.057775,346.806091,350.850419,4.044255,1.773752,3.785028,1.509702,791.073071,2024-02-28 02:54:11.673804800,0.210340
min,1.000000,1.000000,0.000000,0.048000,0.001000,1.000000,1.000000,-1.000000,1.000000,2022-12-31 00:00:00,0.000000
25%,6748.250000,5.000000,57.683500,62.483000,0.500000,1.000000,1.000000,1.000000,999.000000,2023-12-31 00:00:00,0.000000
50%,13570.500000,10.000000,151.453000,155.692000,1.626500,2.000000,1.000000,1.000000,999.000000,2024-07-31 00:00:00,0.000000
75%,20432.500000,15.000000,436.901000,438.541000,4.942000,2.000000,11.000000,2.000000,999.000000,2024-12-31 00:00:00,0.000000
max,27290.000000,25.000000,1805.083000,1806.027000,135.462000,7.000000,11.000000,3.000000,999.000000,2024-12-31 00:00:00,1.000000
std,7886.159143,6.788261,430.320034,430.244110,6.486764,0.767543,4.369117,0.614011,402.892962,NaN,0.407559


In [8]:
# Tabla de métricas del Primer Dataset

# -- Completitud: Porcentaje de valores no nulos
completitud1 = round((1 - df1.isnull().sum().sum() / df2.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad1 = round((1 - df1.duplicated().sum() / len(df2)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales1 = 21
cols_con_problema1 = 3
consistencia1 = round(((cols_totales1 - cols_con_problema1) / cols_totales1) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas1 = pd.DataFrame({
    'Dataset': ["Dataset GEI CSV_0.csv"],
    'Filas': [len(df1)],
    'Completitud': [completitud1],
    'Unicidad': [unicidad1],
    'Consistencia': [consistencia1]
})

print("Métricas de calidad del dataset original:")
tabla_metricas1

Métricas de calidad del dataset original:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Dataset GEI CSV_0.csv,27290,100.0,100.0,85.71


In [9]:
# -- Completitud: Porcentaje de valores no nulos
completitud1 = round((1 - df1_limpio.isnull().sum().sum() / df2.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad1 = round((1 - df1_limpio.duplicated().sum() / len(df2)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales1 = 21
cols_con_problema1 = 0
consistencia1 = round(((cols_totales1 - cols_con_problema1) / cols_totales1) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas1 = pd.DataFrame({
    'Dataset': ["Dataset GEI CSV_0.csv"],
    'Filas': [len(df1_limpio)],
    'Completitud': [completitud1],
    'Unicidad': [unicidad1],
    'Consistencia': [consistencia1]
})

print("Métricas de calidad del dataset modificado:")
tabla_metricas1

Métricas de calidad del dataset modificado:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Dataset GEI CSV_0.csv,22882,100.0,100.0,100.0


# Extracción de Dataset 2

In [10]:
# Tratamiento de Dataset 2
print("\n===== Tratamiento de Datasets ====")
print("\nPerfilado del Dataset 2:")

# Métricas antes de la limpieza:
print(f"Nulos: {df2.isnull().sum().sum()}")
print(f"Duplicados: {df2.duplicated().sum()}")

# Identificación de outliders por IQR
numericos2 = df2.select_dtypes(include='number')

q1_2 = numericos2.quantile(0.25)
q3_2 = numericos2.quantile(0.75)
iqr_2 = q3_2 - q1_2
outliers_2 = ((numericos2 < (q1_2 - 1.5 * iqr_2)) | (numericos2 > (q3_2 + 1.5 * iqr_2))).sum()

print("\nNúmero de outliers por columna:")
outliers_2



===== Tratamiento de Datasets ====

Perfilado del Dataset 2:
Nulos: 1285
Duplicados: 0

Número de outliers por columna:


,0
FECHA_CORTE,0
ANIO,0
DIOXIDO_DE_CARBONO_GGCO2,35
METANO_GGCH4,31
METANO_EQUIVALENTE_GGCO2EQ,31
OXIDO_NITROSO_GGN2O,26
OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ,26
HFC_GGCO2EQ,1
EMISIONES_GEI_GGCO2EQ,42


# Tratamiento de Dataset2

In [11]:
# Copia del DataFrame
df2_limpio = df2.copy()

In [12]:
# Normalización de Formato: FECHA_CORTE y ANIO a datetime
df2_limpio['ANIO'] = pd.to_datetime(df2_limpio['ANIO'].astype(str).str.split('.').str[0].str.strip(), format='%Y', errors='coerce')
df2_limpio['FECHA_CORTE'] = pd.to_datetime(df2_limpio['FECHA_CORTE'].astype(str).str.split('.').str[0].str.strip(), format='%Y%m%d', errors='coerce')

print("1. Nuevo formato asignado:")
print(df2_limpio[['ANIO', 'FECHA_CORTE']].head().dtypes)

1. Nuevo formato asignado:
ANIO           datetime64[ns]
FECHA_CORTE    datetime64[ns]
dtype: object


In [13]:
# Detectar valores nulos
print("\n2. Valores nulos antes del tratamiento:")
print(df2.isnull().sum())


2. Valores nulos antes del tratamiento:
FECHA_CORTE                            1
ANIO                                   1
SECTOR                                 1
CATEGORIA                              1
SUBCATEGORIA                          29
FUENTE_DE_EMISION                     57
IPCC_CLASF                             1
DIOXIDO_DE_CARBONO_GGCO2             132
METANO_GGCH4                         162
METANO_EQUIVALENTE_GGCO2EQ           162
OXIDO_NITROSO_GGN2O                  176
OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ    176
HFC_GGCO2EQ                          351
EMISIONES_GEI_GGCO2EQ                 35
dtype: int64


In [14]:
# Estrategia para nulos:
# Eliminar fila de FECHA_CORTE y EMISIONES_GEI_GGCO2EQ con datos nulos
df2_limpio = df2_limpio.dropna(subset=['FECHA_CORTE', 'EMISIONES_GEI_GGCO2EQ'])

# Reemplazar nulos por "Sin Definir" en columnas SUBCATEGORIA y FUENTE_DE_EMISION
df2_limpio['SUBCATEGORIA'] = df2_limpio['SUBCATEGORIA'].fillna('Sin Definir')
df2_limpio['FUENTE_DE_EMISION'] = df2_limpio['FUENTE_DE_EMISION'].fillna('Sin Definir')

# Reemplazar nulos por 0 en columnas numéricas
df2_limpio['DIOXIDO_DE_CARBONO_GGCO2'] = df2_limpio['DIOXIDO_DE_CARBONO_GGCO2'].fillna(0)
df2_limpio['METANO_GGCH4'] = df2_limpio['METANO_GGCH4'].fillna(0)
df2_limpio['METANO_EQUIVALENTE_GGCO2EQ'] = df2_limpio['METANO_EQUIVALENTE_GGCO2EQ'].fillna(0)
df2_limpio['OXIDO_NITROSO_GGN2O'] = df2_limpio['OXIDO_NITROSO_GGN2O'].fillna(0)
df2_limpio['OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ'] = df2_limpio['OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ'].fillna(0)
df2_limpio['HFC_GGCO2EQ'] = df2_limpio['HFC_GGCO2EQ'].fillna(0)

print("\n2. Valores nulos después del tratamiento:")
print(df2_limpio.isnull().sum())


2. Valores nulos después del tratamiento:
FECHA_CORTE                          0
ANIO                                 0
SECTOR                               0
CATEGORIA                            0
SUBCATEGORIA                         0
FUENTE_DE_EMISION                    0
IPCC_CLASF                           0
DIOXIDO_DE_CARBONO_GGCO2             0
METANO_GGCH4                         0
METANO_EQUIVALENTE_GGCO2EQ           0
OXIDO_NITROSO_GGN2O                  0
OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ    0
HFC_GGCO2EQ                          0
EMISIONES_GEI_GGCO2EQ                0
dtype: int64


In [15]:
# Tratamiento de outliers: reemplazar valores menores a 0 con la mediana
print("\n3. Tratamiento de outliers:")
df2_limpio.loc[df2_limpio['DIOXIDO_DE_CARBONO_GGCO2'] < 0, 'DIOXIDO_DE_CARBONO_GGCO2'] = df2_limpio['DIOXIDO_DE_CARBONO_GGCO2'].median()
print(f"Valor negativo reemplazado por mediana: {df2_limpio['DIOXIDO_DE_CARBONO_GGCO2'].median()}")


3. Tratamiento de outliers:
Valor negativo reemplazado por mediana: 107.05


In [16]:
# Estadísticas descriptivas para columnas numéricas
df2_limpio.describe()

,FECHA_CORTE,ANIO,DIOXIDO_DE_CARBONO_GGCO2,METANO_GGCH4,METANO_EQUIVALENTE_GGCO2EQ,OXIDO_NITROSO_GGN2O,OXIDO_NITROSO_EQUIVALENTE_GGCO2EQ,HFC_GGCO2EQ,EMISIONES_GEI_GGCO2EQ
count,323,323,323.000000,323.000000,323.000000,323.000000,323.000000,323.000000,323.000000
mean,2024-05-21 00:00:00,2011-04-06 00:31:12.445820416,2488.037678,27.873932,780.467121,0.832755,220.761672,6.619783,3464.670093
min,2024-05-21 00:00:00,2000-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-2455.670000
25%,2024-05-21 00:00:00,2005-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,90.310000
50%,2024-05-21 00:00:00,2012-01-01 00:00:00,107.050000,0.010000,0.220000,0.000000,0.500000,0.000000,642.500000
75%,2024-05-21 00:00:00,2016-01-01 00:00:00,953.625000,2.980000,83.540000,0.025000,6.695000,0.000000,2940.000000
max,2024-05-21 00:00:00,2019-01-01 00:00:00,64643.220000,558.060000,15625.810000,25.860000,6853.680000,909.910000,64643.220000
std,NaN,NaN,8139.746839,89.897678,2517.142350,3.677024,974.399093,61.753949,8304.974526


In [17]:
# Tabla de métricas del segundo Dataset

# -- Completitud: Porcentaje de valores no nulos
completitud1 = round((1 - df2.isnull().sum().sum() / df2.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad1 = round((1 - df2.duplicated().sum() / len(df2)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales1 = 14
cols_con_problema1 = 2
consistencia1 = round(((cols_totales1 - cols_con_problema1) / cols_totales1) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas1 = pd.DataFrame({
    'Dataset': ["Dataset GEI CSV_0.csv"],
    'Filas': [len(df2)],
    'Completitud': [completitud1],
    'Unicidad': [unicidad1],
    'Consistencia': [consistencia1]
})

print("Métricas de calidad del dataset original:")
tabla_metricas1

Métricas de calidad del dataset original:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Dataset GEI CSV_0.csv,358,74.36,100.0,85.71


In [18]:
# Tabla de métricas del segundo Dataset

# -- Completitud: Porcentaje de valores no nulos
completitud1 = round((1 - df2_limpio.isnull().sum().sum() / df2_limpio.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad1 = round((1 - df2_limpio.duplicated().sum() / len(df2_limpio)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales1 = 14
cols_con_problema1 = 0
consistencia1 = round(((cols_totales1 - cols_con_problema1) / cols_totales1) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas1 = pd.DataFrame({
    'Dataset': ["Red vial nacional del Sistema Nacional de Carreteras 2022 a 2024"],
    'Filas': [len(df2_limpio)],
    'Completitud': [completitud1],
    'Unicidad': [unicidad1],
    'Consistencia': [consistencia1]
})

print("Métricas de calidad del dataset después del tratamiento:")
tabla_metricas1

Métricas de calidad del dataset después del tratamiento:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Red vial nacional del Sistema Nacional de Carr...,323,100.0,100.0,100.0


# Extracción de Dataset 3

In [19]:
#Tratamiento de Dataset 3
print("\n===== Tratamiento de Datasets ====")
print("\nPerfilado del Dataset 3:")

# Métricas antes de la limpieza:
print(f"Nulos: {df3.isnull().sum().sum()}")
print(f"Duplicados: {df3.duplicated().sum()}")
numericos3 = df3.select_dtypes(include='number')

# Identificación de outliders por IQR
q1_3 = numericos3.quantile(0.25)
q3_3 = numericos3.quantile(0.75)
iqr_3 = q3_3 - q1_3
outliers_3 = ((numericos3 < (q1_3 - 1.5 * iqr_3)) | (numericos3 > (q3_3 + 1.5 * iqr_3))).sum()

print("\nNúmero de outliers por columna:")
outliers_3


===== Tratamiento de Datasets ====

Perfilado del Dataset 3:
Nulos: 0
Duplicados: 0

Número de outliers por columna:


,0
ANIO,0
MES,0
TOTAL CARGA NACIONAL,2
TOTAL CARGA INTERNACIONA,1
TOTAL OPERACION NACIONAL,8
TOTAL OPERACION INTERNACIONAL,0


# Tratamiento de Dataset 3

In [20]:
df3_limpio = df3.copy()
print("\n1. Normalización de formato: ")

#Asignación temporal del tipo str para facilitar conversión
# 1. Aseguramos que ANIO y MES sean strings para combinarlos
# 2. Creamos un formato 'YYYY-MM-01'
df3_limpio['FECHA'] = pd.to_datetime(
    df3_limpio['ANIO'].astype(str) + '-' + df3_limpio['MES'].astype(str) + '-01',
    format='%Y-%m-%d',
    errors='coerce'
)

# Eliminar las columnas ANIO y MES permanentemente
df3_limpio = df3_limpio.drop(['ANIO', 'MES'], axis=1)

print(f"\nNueva columna FECHA: {df3_limpio['FECHA'].dtype}")
print(df3_limpio[['FECHA']].head())


1. Normalización de formato: 

Nueva columna FECHA: datetime64[ns]
       FECHA
0 2019-01-01
1 2019-02-01
2 2019-03-01
3 2019-04-01
4 2019-05-01


In [21]:
# Estadísticas descriptivas para columnas numéricas
df3_limpio.describe()

,TOTAL CARGA NACIONAL,TOTAL CARGA INTERNACIONA,TOTAL OPERACION NACIONAL,TOTAL OPERACION INTERNACIONAL,FECHA
count,84.000000,84.000000,84.000000,84.000000,84
mean,2141.481696,16994.778048,8205.619048,4797.357143,2022-06-16 14:17:08.571428608
min,518.897440,8083.799710,970.000000,516.000000,2019-01-01 00:00:00
25%,1856.462178,15013.038320,7239.500000,3405.250000,2020-09-23 12:00:00
50%,2242.222560,16798.305255,9062.000000,5439.000000,2022-06-16 00:00:00
75%,2420.352250,18466.620592,9831.750000,6537.750000,2024-03-08 18:00:00
max,3141.589000,23221.259000,20725.000000,7289.000000,2025-12-01 00:00:00
std,456.493960,3034.677017,2872.166287,2002.705975,NaN


In [22]:
# Tabla de métricas del tercer Dataset

# -- Completitud: Porcentaje de valores no nulos
completitud3 = round((1 - df3.isnull().sum().sum() / df3.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad3 = round((1 - df3.duplicated().sum() / len(df3)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales3 = 8
cols_con_problema3 = 0
consistencia3 = round(((cols_totales3 - cols_con_problema3) / cols_totales3) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas3 = pd.DataFrame({
    'Dataset': ["Información Trafico - Carga y Operaciones - LAP Ene 2019 - Dic 2025"],
    'Filas': [len(df3)],
    'Completitud': [completitud3],
    'Unicidad': [unicidad3],
    'Consistencia': [consistencia3]
})

print("Métricas de calidad del dataset:")
tabla_metricas3

Métricas de calidad del dataset:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Información Trafico - Carga y Operaciones - LA...,84,100.0,100.0,100.0


In [23]:
# -- Completitud: Porcentaje de valores no nulos
completitud3 = round((1 - df3_limpio.isnull().sum().sum() / df3_limpio.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad3 = round((1 - df3_limpio.duplicated().sum() / len(df3_limpio)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales3 = 7
cols_con_problema3 = 0
consistencia3 = round(((cols_totales3 - cols_con_problema3) / cols_totales3) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas3 = pd.DataFrame({
    'Dataset': ["Información Trafico - Carga y Operaciones - LAP Ene 2019 - Dic 2025"],
    'Filas': [len(df3_limpio)],
    'Completitud': [completitud3],
    'Unicidad': [unicidad3],
    'Consistencia': [consistencia3]
})

print("Métricas de calidad del dataset moodificado:")
tabla_metricas3

Métricas de calidad del dataset moodificado:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Información Trafico - Carga y Operaciones - LA...,84,100.0,100.0,100.0


# Extracción de Dataset 4

In [24]:
#Tratamiento de Dataset 4
print("\n===== Tratamiento de Datasets ====")
print("\nPerfilado del Dataset 4:")

# Métricas antes de la limpieza:
print(f"Nulos:\n{df4.isnull().sum()}")
print(f"Duplicados:\n{df4.duplicated().sum()}")
numericos4 = df4.select_dtypes(include='number')

# Identificación de outliders por IQR
q1_4 = numericos4.quantile(0.25)
q3_4 = numericos4.quantile(0.75)
iqr_4 = q3_4 - q1_4
outliers_4 = ((numericos4 < (q1_4 - 1.5 * iqr_4)) | (numericos4 > (q3_4 + 1.5 * iqr_4))).sum()

print("\nNúmero de outliers por columna:")
outliers_4


===== Tratamiento de Datasets ====

Perfilado del Dataset 4:
Nulos:
ID                      0
RUC                     0
RAZON_SOCIAL            0
FECHA_RESOLUCION        0
VIGENCIA_HASTA          0
MOTIVO_HABILITAC        0
TENENCIA                0
PERMISO_BLOQUE          0
PLACA                   0
TIPO_SERVICIO           0
CLASE                   0
N_EJES                  0
ANIO_FAB            12238
N_CHASIS            12238
N_MOTOR             12256
MARCA               12238
COMBUSTIBLE         12238
CARGA_UTIL          12238
P_SECO              12238
P_BRUTO             12238
LARGO               12238
ANCHO               12238
ALTO                12238
N_ASIENTOS          12251
N_LLANTAS           12251
FECHA DE CORTE          0
dtype: int64
Duplicados:
0

Número de outliers por columna:


,0
ID,0
RUC,0
FECHA_RESOLUCION,226
VIGENCIA_HASTA,28
N_EJES,3278
ANIO_FAB,547
CARGA_UTIL,37
P_SECO,714
P_BRUTO,91
LARGO,0


# Tratamiento de Dataset4

In [25]:
# --- 0. DIAGNÓSTICO INICIAL (ANTES) ---
print("=== ESTADO INICIAL DEL DATASET ===")

cols_criticas = ['N_CHASIS', 'N_MOTOR', 'MARCA', 'COMBUSTIBLE',
                 'CARGA_UTIL', 'P_SECO', 'P_BRUTO', 'LARGO', 'ANCHO',
                 'ALTO', 'N_ASIENTOS', 'N_LLANTAS']

print("\nNulos iniciales en columnas críticas:")
print(df4[cols_criticas].isnull().sum())

=== ESTADO INICIAL DEL DATASET ===

Nulos iniciales en columnas críticas:
N_CHASIS       12238
N_MOTOR        12256
MARCA          12238
COMBUSTIBLE    12238
CARGA_UTIL     12238
P_SECO         12238
P_BRUTO        12238
LARGO          12238
ANCHO          12238
ALTO           12238
N_ASIENTOS     12251
N_LLANTAS      12251
dtype: int64


In [26]:
# --- 1. PROCESAMIENTO ETL ---
df4_limpio = df4.copy()

# A. Tratamiento de Columnas Numéricas
cols_num = ['CARGA_UTIL', 'P_SECO', 'P_BRUTO', 'LARGO', 'ANCHO', 'ALTO', 'N_ASIENTOS', 'N_LLANTAS']
for col in cols_num:
    df4_limpio[col] = pd.to_numeric(df4_limpio[col], errors='coerce')

print("\nTipo de dato actual:",df4_limpio[cols_num].dtypes)


Tipo de dato actual: CARGA_UTIL    float64
P_SECO        float64
P_BRUTO       float64
LARGO         float64
ANCHO         float64
ALTO          float64
N_ASIENTOS    float64
N_LLANTAS     float64
dtype: object


In [27]:
# Tratamiento de nulos: Eliminación de columnas innecesarias
df4_limpio = df4_limpio.drop(['LARGO', 'ANCHO', "ALTO", "N_LLANTAS", "N_ASIENTOS"], axis=1)

In [28]:
# Rellenar P_BRUTO usando la mediana de su categoría (ej. CLASE_VEHICULO)
df4_limpio['P_BRUTO'] = df4_limpio.groupby('CLASE')['P_BRUTO'].transform(lambda x: x.fillna(x.median()))

# Repetir para los otros pesos si son necesarios
df4_limpio['P_SECO'] = df4_limpio.groupby('CLASE')['P_SECO'].transform(lambda x: x.fillna(x.median()))

# Repetir para los otros pesos si son necesarios
df4_limpio['CARGA_UTIL'] = df4_limpio.groupby('CLASE')['CARGA_UTIL'].transform(lambda x: x.fillna(x.median()))

# C. Tratamiento de Columnas de Texto
cols_texto = ['N_CHASIS', 'N_MOTOR', 'MARCA', 'COMBUSTIBLE']
for col in cols_texto:
    df4_limpio[col] = df4_limpio[col].fillna('Sin Registro').astype(str).str.strip()

print("\n2. Verificación de consistencia en registros:")
print(df4_limpio[['CARGA_UTIL', 'P_SECO', 'P_BRUTO']].head(10))

print("\n3.Nulos después de tratamiento en columnas de texto:")
print(df4_limpio[cols_texto].isnull().sum())


2. Verificación de consistencia en registros:
   CARGA_UTIL  P_SECO  P_BRUTO
0        30.0     7.5     38.4
1        30.0     7.5     38.4
2        30.0     7.5     38.4
3        30.0     7.5     38.4
4        17.7     8.1     26.2
5        30.0     7.5     38.4
6        30.0     7.5     38.4
7        13.0     7.3     23.0
8        17.7     8.1     26.2
9        30.0     7.5     38.4

3.Nulos después de tratamiento en columnas de texto:
N_CHASIS       0
N_MOTOR        0
MARCA          0
COMBUSTIBLE    0
dtype: int64


In [29]:
# D. Unificación de categorías (Combustible y Motor)
print("\n4. [COMBUSTIBLE] Categorías antes de unificar (Top 10):")
print(df4_limpio['COMBUSTIBLE'].value_counts().head(10))

ausencia_combustible = ["SIN COMBUSTIBLE", "SIN DEFINIR", "NO MUESTRA", "SIN VALOR", "S/C", "NAN", "NONE"]
df4_limpio['COMBUSTIBLE'] = df4_limpio['COMBUSTIBLE'].replace(ausencia_combustible, "Sin Registro")
df4_limpio['COMBUSTIBLE'] = df4_limpio['COMBUSTIBLE'].replace("BI - COMBUSTIBLE GLP", "BI-COMBUSTIBLE GLP")

# 1. COMBUSTIBLE: Resultado
print("\n5. [COMBUSTIBLE] Categorías después de unificar:")
print(df4_limpio['COMBUSTIBLE'].value_counts())



4. [COMBUSTIBLE] Categorías antes de unificar (Top 10):
COMBUSTIBLE
Sin Registro       12238
S/C                 5760
DIESEL              5310
PETROLEO             284
DIÉSEL               242
GNV                   32
SIN COMBUSTIBLE       30
SIN DEFINIR           25
GNL                    9
NO MUESTRA             7
Name: count, dtype: int64

5. [COMBUSTIBLE] Categorías después de unificar:
COMBUSTIBLE
Sin Registro          18065
DIESEL                 5310
PETROLEO                284
DIÉSEL                  242
GNV                      32
GNL                       9
BI-COMBUSTIBLE GLP        4
GASOLINA                  3
DUAL GLP                  1
Name: count, dtype: int64


In [30]:
# 2. N_MOTOR: Diagnóstico previo
print("\n" + "-"*30)
print("\n6. [N_MOTOR] Valores atípicos antes (Top 10):")
print(df4_limpio['N_MOTOR'].value_counts().head(10))

ausencia_motor = ["S", "NE", "NOTIENE", "00000", "00", "---", "-", "NAN"]
df4_limpio['N_MOTOR'] = df4_limpio['N_MOTOR'].replace(ausencia_motor, "Sin Registro")

# 2. N_MOTOR: Resultado
print("\n[7. N_MOTOR] Valores después de estandarizar (Top 5):")
print(df4_limpio['N_MOTOR'].value_counts().head(5))

# --- 2. VERIFICACIÓN DETALLADA (DESPUÉS) ---
print("\n" + "="*40)
print("=== ESTADO FINAL DEL DATASET ===")
print("="*40)

print("\n8. Conteo final de 'Sin Registro' en columnas clave:")
for col in ['MARCA', 'COMBUSTIBLE']:
    conteo = (df4_limpio[col] == 'Sin Registro').sum()
    print(f"- {col}: {conteo} registros")


------------------------------

6. [N_MOTOR] Valores atípicos antes (Top 10):
N_MOTOR
Sin Registro     12256
-                 5227
0                  430
S                   91
NOTIENE             22
NE                  11
00000               10
00                   8
SINMOTOR             6
NOCORRESPONDE        4
Name: count, dtype: int64

[7. N_MOTOR] Valores después de estandarizar (Top 5):
N_MOTOR
Sin Registro     17625
0                  430
SINMOTOR             6
NOCORRESPONDE        4
SSINMOTOR            2
Name: count, dtype: int64

=== ESTADO FINAL DEL DATASET ===

8. Conteo final de 'Sin Registro' en columnas clave:
- MARCA: 12238 registros
- COMBUSTIBLE: 18065 registros


In [31]:
# 1. Valores de fecha antes del tratamiento
print("9. Formato de fechas original (ejemplos):")
print(df4[['FECHA_RESOLUCION', 'VIGENCIA_HASTA', 'FECHA DE CORTE', "ANIO_FAB"]].head())

# --- CONVERSIÓN A DATETIME ---
# Limpiamos decimales .0 y espacios antes de convertir
# usando una tupla ('FECHA_RESOLUCION', 'VIGENCIA_HASTA', 'FECHA DE CORTE') como la clave.
# Pandas interpreta esto como intentar encontrar una sola columna con una tupla como nombre, lo cual no existe.
# Para aplicar la operación a varias columnas, se itera sobre ellas como una lista.

columns_to_convert = ['FECHA_RESOLUCION', 'VIGENCIA_HASTA', 'FECHA DE CORTE']
for col_name in columns_to_convert:
    df4_limpio[col_name] = pd.to_datetime(
        df4_limpio[col_name].astype(str).str.split('.').str[0].str.strip(),
        format='%Y%m%d',
        errors='coerce'
    )

df4_limpio['ANIO_FAB'] = pd.to_datetime(df4_limpio['ANIO_FAB'].astype(str).str.split('.').str[0].str.strip(), format='%Y', errors='coerce')

# --- AJUSTE EN EL PROCESAMIENTO ETL PARA ORDENAR COLUMNAS ---

# En lugar de crear la columna al final, buscamos su posición e insertamos a la derecha (+1)
# 1. Calculamos el estado
estado_values = df4_limpio["ANIO_FAB"].apply(lambda x: 'Registrado' if pd.notnull(x) else 'Sin Registro')

# 2. Obtenemos la posición actual de la columna original
idx = df4_limpio.columns.get_loc("ANIO_FAB")

# 3. Insertamos la nueva columna justo en la posición de la derecha (idx + 1)
# Usamos un try/except por si se ejecuta el código dos veces y la columna ya existe
nombre_col_estado = f'{"ANIO_FAB"}_ESTADO'
if nombre_col_estado not in df4_limpio.columns:
   df4_limpio.insert(loc=idx + 1, column=nombre_col_estado, value=estado_values)
else:
  # Si ya existe (segunda ejecución), solo actualizamos los valores
   df4_limpio[nombre_col_estado] = estado_values

# 2. Verificación final
print("\n10. Fechas después del tratamiento y Clasificación:")
# Mostramos las fechas junto a la nueva columna de identificación
cols_verificar = ['FECHA_RESOLUCION','VIGENCIA_HASTA', 'FECHA DE CORTE', "ANIO_FAB", "ANIO_FAB_ESTADO"]
print(df4_limpio[cols_verificar].head(10))

print("\n11. Tipos de datos finales:")
print(df4_limpio[['FECHA_RESOLUCION', 'VIGENCIA_HASTA', 'FECHA DE CORTE', "ANIO_FAB", "ANIO_FAB_ESTADO"]].dtypes)


9. Formato de fechas original (ejemplos):
   FECHA_RESOLUCION  VIGENCIA_HASTA  FECHA DE CORTE  ANIO_FAB
0          20201021        20251021        20231231       NaN
1          20201021        20251021        20231231       NaN
2          20201021        20251021        20231231       NaN
3          20201130        20251021        20231231       NaN
4          20211109        20311109        20231231       NaN

10. Fechas después del tratamiento y Clasificación:
  FECHA_RESOLUCION VIGENCIA_HASTA FECHA DE CORTE ANIO_FAB ANIO_FAB_ESTADO
0       2020-10-21     2025-10-21     2023-12-31      NaT    Sin Registro
1       2020-10-21     2025-10-21     2023-12-31      NaT    Sin Registro
2       2020-10-21     2025-10-21     2023-12-31      NaT    Sin Registro
3       2020-11-30     2025-10-21     2023-12-31      NaT    Sin Registro
4       2021-11-09     2031-11-09     2023-12-31      NaT    Sin Registro
5       2022-04-25     2025-08-03     2023-12-31      NaT    Sin Registro
6       2013-01

In [32]:
# Estadísticas básicas
print("\n12. Estadísticas descriptivas:")
print(df4_limpio.describe())

print("\n" + "="*50 + "\n")


12. Estadísticas descriptivas:
                ID           RUC               FECHA_RESOLUCION  \
count  23950.00000  2.395000e+04                          23950   
mean   11975.50000  2.043988e+10  2020-04-14 09:53:33.394571776   
min        1.00000  2.010019e+10            1996-04-10 00:00:00   
25%     5988.25000  2.037133e+10            2018-02-02 00:00:00   
50%    11975.50000  2.050604e+10            2021-06-21 00:00:00   
75%    17962.75000  2.055436e+10            2022-09-02 00:00:00   
max    23950.00000  2.061331e+10            2024-12-27 00:00:00   
std     6913.91381  1.712811e+08                            NaN   

                      VIGENCIA_HASTA        N_EJES  \
count                          23935  23950.000000   
mean   2029-05-21 14:17:19.314810880      2.902589   
min              2024-01-04 00:00:00      1.000000   
25%              2026-10-25 00:00:00      3.000000   
50%              2029-02-27 00:00:00      3.000000   
75%              2031-12-16 00:00:00    

In [33]:
# Tabla de métricas del cuarto Dataset

# -- Completitud: Porcentaje de valores no nulos
completitud4 = round((1 - df4.isnull().sum().sum() / df4.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad4 = round((1 - df4.duplicated().sum() / len(df4)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales4 = 26
cols_con_problema4 = 13
consistencia4 = round(((cols_totales4 - cols_con_problema4) / cols_totales4) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas4 = pd.DataFrame({
    'Dataset': ["Transporte Terrestre Carga Internacional_2023-2024"],
    'Filas': [len(df4)],
    'Completitud': [completitud4],
    'Unicidad': [unicidad4],
    'Consistencia': [consistencia4]
})

print("Métricas de calidad del dataset:")
tabla_metricas4

Métricas de calidad del dataset:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Transporte Terrestre Carga Internacional_2023-...,23950,74.44,100.0,50.0


In [34]:
# -- Completitud: Porcentaje de valores no nulos
completitud4 = round((1 - df4_limpio.isnull().sum().sum() / df4_limpio.size) * 100, 2)

# -- Unicidad: Porcentaje de filas únicas
unicidad4 = round((1 - df4_limpio.duplicated().sum() / len(df4_limpio)) * 100, 2)

# -- Consistencia: Porcentaje de columnas sin problemas de rango o formato
cols_totales4 = 27
cols_con_problema4 = 1
consistencia4 = round(((cols_totales4 - cols_con_problema4) / cols_totales4) * 100, 2)

# Creación de la tabla de métricas
tabla_metricas4 = pd.DataFrame({
    'Dataset': ["Transporte Terrestre Carga Internacional_2023-2024"],
    'Filas': [len(df4_limpio)],
    'Completitud': [completitud4],
    'Unicidad': [unicidad4],
    'Consistencia': [consistencia4]
})

print("Métricas de calidad del dataset:")
tabla_metricas4

Métricas de calidad del dataset:


,Dataset,Filas,Completitud,Unicidad,Consistencia
0,Transporte Terrestre Carga Internacional_2023-...,23950,97.67,100.0,96.3


Load de Datasets

In [35]:
# Guardar como CSV (Standard)
df1_limpio.to_csv('Limpio - Red vial nacional del Sistema Nacional de Carreteras 2022 a 2024 - Ministerio de Transportes y Comunicaciones – MTC.csv', index=False, encoding='utf-8-sig')
df2_limpio.to_csv('Limpio - Dataset GEI CSV_0.csv', index=False, encoding='utf-8-sig')
df3_limpio.to_csv('Limpio - Información Trafico - Carga y Operaciones - LAP Ene 2019 - Dic 2025 - Organismo Supervisor de la Inversión en Infraestructura de Transporte de Uso Público – OSITRAN.csv', index=False, encoding='utf-8-sig')
df4_limpio.to_csv('Limpio - Transporte Terrestre Carga Internacional_2023-2024.csv', index=False, encoding='utf-8-sig')